In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os  
# Load environment variables from .env file
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}",
    echo=True
)
from sqlalchemy import text
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS staging;"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS core;"))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS core.sales_records (
        order_id BIGINT PRIMARY KEY,
        region TEXT,
        country TEXT,
        item_type TEXT,
        sales_channel TEXT,
        order_priority TEXT,
        order_date DATE,
        ship_date DATE,
        units_sold DOUBLE PRECISION,
        unit_price DOUBLE PRECISION,
        unit_cost DOUBLE PRECISION,
        total_revenue DOUBLE PRECISION,
        total_cost DOUBLE PRECISION,
        total_profit DOUBLE PRECISION
    );"""))
# 1) Extract
rawdata = pd.read_csv(r"C:\Users\SHEHARYAR HAMD\OneDrive\Desktop\10000 Sales Records.csv")
rawdata.to_sql(
    "sales_records_raw",
    engine,
    schema="staging",
    if_exists="replace",
    index=False
)
data=rawdata.copy()
# 2) Transform (fix nulls)
data["Unit Cost"] = data["Unit Cost"].fillna(data["Unit Cost"].mean())
data["Units Sold"] = data["Units Sold"].fillna(data["Units Sold"].mean())
data["Order Date"] = data["Order Date"].fillna("2020-01-01")

# parse dates properly
data["Order Date"] = data["Order Date"].fillna("2020-01-01")
data["Order Date"] = pd.to_datetime(data["Order Date"], format="mixed", errors="coerce")
data["Ship Date"] = pd.to_datetime(data["Ship Date"], format="mixed", errors="coerce")
# rename columns to be SQL-friendly
data.columns = (
    data.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# 3) Load (creates table automatically if it doesn't exist)
data.to_sql(
    "sales_records",
    engine,
    if_exists="replace", 
    index=False,
    schema="core",
    chunksize=5000,
    method="multi"
)

print("Loaded successfully ✅")

In [ ]:
upsert_sql = """
INSERT INTO core.sales_records (
    order_id, region, country, item_type, sales_channel, order_priority,
    order_date, ship_date, units_sold, unit_price, unit_cost,
    total_revenue, total_cost, total_profit
)
SELECT
    order_id, region, country, item_type, sales_channel, order_priority,
    order_date, ship_date, units_sold, unit_price, unit_cost,
    total_revenue, total_cost, total_profit
FROM staging.sales_records_raw
ON CONFLICT (order_id) DO UPDATE SET
    region = EXCLUDED.region,
    country = EXCLUDED.country,
    item_type = EXCLUDED.item_type,
    sales_channel = EXCLUDED.sales_channel,
    order_priority = EXCLUDED.order_priority,
    order_date = EXCLUDED.order_date,
    ship_date = EXCLUDED.ship_date,
    units_sold = EXCLUDED.units_sold,
    unit_price = EXCLUDED.unit_price,
    unit_cost = EXCLUDED.unit_cost,
    total_revenue = EXCLUDED.total_revenue,
    total_cost = EXCLUDED.total_cost,
    total_profit = EXCLUDED.total_profit;
"""

from sqlalchemy import text

try:
    with engine.begin() as conn:
        conn.execute(text(upsert_sql))
    print("UPSERT done")
except Exception as e:
    print("UPSERT failed")
    print(e)